### **Hyperparameter Tuning the ANN using Optuna**

- Hyperparmeter Tuning on Parameters:
    1) Number of Hidden Layers
    2) Neurons Per Layer
    3) Number of Epochs
    4) Optimizer
    5) Learning Rate
    6) Batch Size
    7) Dropout Rate
    8) Weight Decay (lambda)

- Use of optuna
    1) create objective function
        - `def objective()`
        - search space
        - model init
        - parameters init
        - training loop
        - evaluation loop
    2) Study Object
        - Trail
        

In [26]:
%pip install optuna

In [27]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import optuna

In [28]:
torch.manual_seed(42)

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device is {device}")

Device is cuda


In [30]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
dataset_path = "/content/drive/MyDrive/DataSets/fashion-mnist_train.csv"

In [32]:
df = pd.read_csv(dataset_path)

In [33]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [34]:
df.shape

(60000, 785)

In [35]:
X = df.drop(columns="label")
y = df["label"]

In [36]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [37]:
# scaling 
X_train = X_train / 255.0
X_test = X_test / 255.0

In [38]:
len(X_train)

48000

In [39]:
# CustomDatset class
class CustomDatset(Dataset):
    def __init__(self,features, labels):
        self.features = torch.tensor(features.to_numpy(),dtype=torch.float32)
        self.labels = torch.tensor(labels.to_numpy(),dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self,idx):
        return self.features[idx], self.labels[idx]

In [40]:
# creating training dataset and testing dataset object from CustomDatset class
training_dataset = CustomDatset(X_train,y_train)
testing_datset = CustomDatset(X_test,y_test)

In [41]:
class MyANN(nn.Module):
    def __init__(self,input_dim,output_dim,num_hidden_layer,neurons_per_layer,dropout_rate):
        super().__init__()
        layers = []
        for i in range(num_hidden_layer):
            layers.append(nn.Linear(input_dim,neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            input_dim = neurons_per_layer
        layers.append(nn.Linear(neurons_per_layer,output_dim))

        self.model = nn.Sequential(*layers)

    def forward(self,x):
        return self.model(x)

In [42]:
# objective function
def objective(trail):
    # next trail hyperparameter values from search space
    num_hidden_layer = trail.suggest_int("num_hidden_layers",1,5)
    neurons_per_layer = trail.suggest_int("neurons_per_layer",8,128,step=8)
    epochs = trail.suggest_int("epochs",10,50,step=10)
    learning_rate = trail.suggest_float("learning_rate",1e-5,1e-1,log=True)
    dropout_rate = trail.suggest_float("dropout_rate",0.1,0.5,step=0.1)
    batch_size = trail.suggest_categorical("batch_size",[16,32,64,128])
    optimizer_name = trail.suggest_categorical("optimizer",["Adam","SGD","RMSprop"])
    weight_decay = trail.suggest_float("weight_decay",1e-5,1e-3,log=True)

    # creating test loader and train loader object from DataLoader Class
    train_loader = DataLoader(training_dataset,batch_size=batch_size,shuffle=True, pin_memory=True)
    test_loader = DataLoader(testing_datset,batch_size=batch_size,shuffle=False,pin_memory=False)

    

    # model init
    input_dim = 784
    output_dim = 10
    model = MyANN(input_dim,output_dim,num_hidden_layer,neurons_per_layer,dropout_rate)
    model.to(device)

    
    
    # optimizer selection
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(),lr=learning_rate,weight_decay=weight_decay)

    if optimizer_name == "Adam":
        optim.Adam(model.parameters(),lr=learning_rate,weight_decay= weight_decay)
    elif optimizer_name == "SGD":
        optim.SGD(model.parameters(),lr=learning_rate,weight_decay= weight_decay)
    else:
        optim.RMSprop(model.parameters(),lr=learning_rate,weight_decay= weight_decay)
    
    # training loop
    for epoch in range(epochs):
        for batch_features, batch_labels in train_loader:
            # move data to gpu
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

            # forward pass
            output = model(batch_features)

            # calculating loss
            loss = criterion(output,batch_labels)

            # back pass
            optimizer.zero_grad()
            loss.backward()

            # gradients update
            optimizer.step()
        
        

    # evaluation
    model.eval()

    total = 0
    correct = 0

    with torch.no_grad():
        for batch_features,batch_labels in test_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            outputs = model(batch_features)
            _, predicted = torch.max(outputs,1)

            total = total + batch_labels.shape[0]
            correct = correct + (predicted == batch_labels).sum().item()
        accuracy = correct/total
        return accuracy


In [43]:
study = optuna.create_study(direction="maximize")

[I 2026-05-29 11:24:53,381] A new study created in memory with name: no-name-e07ff62a-6d05-4a1a-9589-29a343f8310a


In [44]:
study.optimize(objective,n_trials=5)

[I 2026-05-29 11:33:42,957] Trial 0 finished with value: 0.8586666666666667 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 104, 'epochs': 50, 'learning_rate': 0.000265409600988439, 'dropout_rate': 0.30000000000000004, 'batch_size': 16, 'optimizer': 'RMSprop', 'weight_decay': 2.9978877700263082e-05}. Best is trial 0 with value: 0.8586666666666667.
[I 2026-05-29 11:35:15,241] Trial 1 finished with value: 0.8080833333333334 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 72, 'epochs': 20, 'learning_rate': 0.0002911721854347874, 'dropout_rate': 0.30000000000000004, 'batch_size': 32, 'optimizer': 'RMSprop', 'weight_decay': 2.3503756488998622e-05}. Best is trial 0 with value: 0.8586666666666667.
[I 2026-05-29 11:36:43,651] Trial 2 finished with value: 0.6788333333333333 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 8, 'epochs': 40, 'learning_rate': 0.004966378824149833, 'dropout_rate': 0.1, 'batch_size': 128, 'optimizer': 'Adam', 'weight_decay': 0

In [45]:
study.best_value

0.8753333333333333

In [46]:
study.best_params

{'num_hidden_layers': 3,
 'neurons_per_layer': 128,
 'epochs': 40,
 'learning_rate': 0.00019512233613987846,
 'dropout_rate': 0.1,
 'batch_size': 16,
 'optimizer': 'RMSprop',
 'weight_decay': 0.0005177271154003942}